# 99. 실습 리소스 정리

`lab_state.json`에 기록된 리소스를 역순(실습 4 → 3 → 1)으로 삭제합니다. 셀을 위에서부터 차례로 실행하세요. 이미 지워진 리소스는 건너뜁니다.

> VPC 인터페이스 엔드포인트, 게이트웨이, KMS 키는 시간·월 단위 요금이 발생합니다. 실습이 끝나면 바로 정리하세요. CloudFormation이 삭제하는 KMS 키는 즉시 지워지지 않고 삭제 대기(기본 30일) 상태가 됩니다.

In [ ]:
import sys, json, time
sys.path.append("..")
from common.labutils import *
state = load_state()
iam = client("iam")

def safe(label, fn, *a, **k):
    try:
        fn(*a, **k); print("삭제:", label)
    except Exception as e:
        print("건너뜀:", label, "-", type(e).__name__, str(e)[:100])

def delete_role(name):
    try:
        for p in iam.list_role_policies(RoleName=name)["PolicyNames"]:
            iam.delete_role_policy(RoleName=name, PolicyName=p)
        for p in iam.list_attached_role_policies(RoleName=name)["AttachedPolicies"]:
            iam.detach_role_policy(RoleName=name, PolicyArn=p["PolicyArn"])
        iam.delete_role(RoleName=name); print("삭제: 역할", name)
    except Exception as e:
        print("건너뜀: 역할", name, "-", str(e)[:80])
state

## 실습 4: AgentCore

In [ ]:
ctl, logs = client("bedrock-agentcore-control"), client("logs")
gw_id, pe_id = state.get("gateway_id"), state.get("policy_engine_id")

if state.get("gateway_log_delivery"):
    name = state["gateway_log_delivery"]
    for d in logs.describe_deliveries().get("deliveries", []):
        if d.get("deliverySourceName") == f"{name}-src":
            safe("로그 전달", logs.delete_delivery, id=d["id"])
    safe("로그 전달 소스", logs.delete_delivery_source, name=f"{name}-src")
    safe("로그 전달 대상", logs.delete_delivery_destination, name=f"{name}-dst")
    safe("로그 그룹", logs.delete_log_group, logGroupName=state["gateway_log_group"])

if gw_id:
    for t in ctl.list_gateway_targets(gatewayIdentifier=gw_id).get("items", []):
        safe("게이트웨이 타깃", ctl.delete_gateway_target, gatewayIdentifier=gw_id, targetId=t["targetId"])
    time.sleep(10)
    safe("게이트웨이", ctl.delete_gateway, gatewayIdentifier=gw_id)
    time.sleep(15)

if pe_id:
    for p in ctl.list_policies(policyEngineId=pe_id).get("policies", []):
        safe(f"정책 {p['name']}", ctl.delete_policy, policyEngineId=pe_id, policyId=p["policyId"])
    time.sleep(10)
    safe("정책 엔진", ctl.delete_policy_engine, policyEngineId=pe_id)

if state.get("api_key_provider"):
    safe("API 키 자격 증명 공급자", ctl.delete_api_key_credential_provider, name=state["api_key_provider"])
if state.get("workload_identity"):
    safe("워크로드 아이덴티티", ctl.delete_workload_identity, name=state["workload_identity"])
if state.get("emr_function"):
    safe("EMR Lambda", client("lambda").delete_function, FunctionName=state["emr_function"])
if state.get("cognito_pool"):
    safe("Cognito 사용자 풀", client("cognito-idp").delete_user_pool, UserPoolId=state["cognito_pool"])
for r in [state.get("gateway_role"), state.get("emr_function_role")]:
    if r: delete_role(r)

## 실습 3: RAG 보안

In [ ]:
agent, s3, s3v, cfn = client("bedrock-agent"), client("s3"), client("s3vectors"), client("cloudformation")
kb_id = state.get("kb_id")
if kb_id:
    if state.get("kb_data_source"):
        safe("데이터 소스", agent.delete_data_source, knowledgeBaseId=kb_id, dataSourceId=state["kb_data_source"])
        time.sleep(10)
    safe("지식 기반", agent.delete_knowledge_base, knowledgeBaseId=kb_id)
    time.sleep(20)
if state.get("vector_bucket"):
    safe("벡터 인덱스", s3v.delete_index, vectorBucketName=state["vector_bucket"], indexName=state["vector_index"])
    safe("벡터 버킷", s3v.delete_vector_bucket, vectorBucketName=state["vector_bucket"])
if state.get("kb_role"):
    delete_role(state["kb_role"])

stack = state.get("rag_stack")
if stack:
    try:
        out = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=stack)["Stacks"][0]["Outputs"]}
        bucket = out["DocsBucketName"]
        # 버전 관리 버킷은 모든 버전과 삭제 마커를 지워야 비울 수 있습니다
        pages = s3.get_paginator("list_object_versions").paginate(Bucket=bucket)
        for page in pages:
            objs = [{"Key": v["Key"], "VersionId": v["VersionId"]} for v in page.get("Versions", []) + page.get("DeleteMarkers", [])]
            if objs:
                s3.delete_objects(Bucket=bucket, Delete={"Objects": objs})
        print("버킷 비움:", bucket)
    except Exception as e:
        print("버킷 비우기 건너뜀:", str(e)[:100])
    cfn.delete_stack(StackName=stack)
    print("스택 삭제 시작:", stack, "(VPC 엔드포인트·Lambda ENI 정리에 5~15분 걸릴 수 있음)")
    cfn.get_waiter("stack_delete_complete").wait(StackName=stack, WaiterConfig={"Delay": 20, "MaxAttempts": 60})
    print("스택 삭제 완료")

## 실습 1: Guardrail

In [ ]:
if state.get("guardrail_id"):
    safe("Guardrail(모든 버전)", client("bedrock").delete_guardrail, guardrailIdentifier=state["guardrail_id"])

In [ ]:
STATE_FILE.unlink(missing_ok=True)
print("정리 완료. lab_state.json 삭제됨")